# The Protocol Zoo — A2A, MCP, and Multi-Agent Communication

Companion notebook for the blog post: [The Protocol Zoo](https://cmenguy.github.io/llm/ai-engineering/2026/01/28/agentic-protocols.html)

This notebook demonstrates the core concepts of the two major agentic protocols:
- **MCP (Model Context Protocol)** — giving an AI app access to tools and data
- **A2A (Agent-to-Agent)** — enabling agents to discover and collaborate with each other

We'll build working examples of agent cards, tool definitions, task lifecycles, and communication patterns.

## Setup

Install dependencies. We use `pydantic` for data modeling and `httpx` for async HTTP.

In [1]:
!pip install -q pydantic httpx


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


## Part 1: MCP — Tool Definitions and the Context Layer

MCP's core primitives are **tools**, **resources**, and **prompts**. Let's model them.

In [2]:
from pydantic import BaseModel, Field
from typing import Any
import json


class MCPTool(BaseModel):
    """An MCP tool definition — a function the AI can call."""
    name: str
    description: str
    inputSchema: dict[str, Any]


class MCPResource(BaseModel):
    """An MCP resource — data the AI can access for context."""
    uri: str
    name: str
    mimeType: str
    description: str = ""
    annotations: dict[str, Any] = Field(default_factory=dict)


# Define some example MCP tools
lookup_tool = MCPTool(
    name="lookup_user",
    description="Look up a user by email address",
    inputSchema={
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "User email"}
        },
        "required": ["email"],
    },
)

weather_tool = MCPTool(
    name="get_weather",
    description="Get current weather for a location",
    inputSchema={
        "type": "object",
        "properties": {
            "location": {"type": "string", "description": "City or zip code"}
        },
        "required": ["location"],
    },
)

print("=== MCP Tool: lookup_user ===")
print(json.dumps(lookup_tool.model_dump(), indent=2))
print()
print("=== MCP Tool: get_weather ===")
print(json.dumps(weather_tool.model_dump(), indent=2))

=== MCP Tool: lookup_user ===
{
  "name": "lookup_user",
  "description": "Look up a user by email address",
  "inputSchema": {
    "type": "object",
    "properties": {
      "email": {
        "type": "string",
        "description": "User email"
      }
    },
    "required": [
      "email"
    ]
  }
}

=== MCP Tool: get_weather ===
{
  "name": "get_weather",
  "description": "Get current weather for a location",
  "inputSchema": {
    "type": "object",
    "properties": {
      "location": {
        "type": "string",
        "description": "City or zip code"
      }
    },
    "required": [
      "location"
    ]
  }
}


### MCP Capability Negotiation

When an MCP client connects to a server, they negotiate capabilities during initialization. The server declares what it supports (tools, resources, prompts) and the client declares what it can handle (sampling, elicitation).

In [3]:
class MCPServerCapabilities(BaseModel):
    """What an MCP server advertises during initialization."""
    tools: dict[str, Any] = Field(default_factory=lambda: {"listChanged": True})
    resources: dict[str, Any] = Field(
        default_factory=lambda: {"subscribe": True, "listChanged": True}
    )
    prompts: dict[str, Any] = Field(default_factory=lambda: {"listChanged": True})


class MCPInitResponse(BaseModel):
    """Server's response to the initialize request."""
    protocolVersion: str = "2025-06-18"
    capabilities: MCPServerCapabilities = Field(
        default_factory=MCPServerCapabilities
    )
    serverInfo: dict[str, str] = Field(
        default_factory=lambda: {"name": "example-server", "version": "1.0.0"}
    )


# Simulate initialization
init_response = MCPInitResponse()
print("=== MCP Init Response ===")
print(json.dumps(init_response.model_dump(), indent=2))

=== MCP Init Response ===
{
  "protocolVersion": "2025-06-18",
  "capabilities": {
    "tools": {
      "listChanged": true
    },
    "resources": {
      "subscribe": true,
      "listChanged": true
    },
    "prompts": {
      "listChanged": true
    }
  },
  "serverInfo": {
    "name": "example-server",
    "version": "1.0.0"
  }
}


### Simulating an MCP Tool Call

In a real MCP flow, the LLM decides to call a tool, the host routes it to the right MCP server, and the result comes back. Let's simulate that loop.

In [4]:
from typing import Callable

# Registry of tools and their implementations
TOOL_REGISTRY: dict[str, tuple[MCPTool, Callable]] = {}


def register_tool(tool: MCPTool, handler: Callable):
    TOOL_REGISTRY[tool.name] = (tool, handler)


def call_mcp_tool(name: str, arguments: dict) -> dict:
    """Simulate an MCP tool call — host routes to the right server."""
    if name not in TOOL_REGISTRY:
        return {"error": f"Unknown tool: {name}"}
    _, handler = TOOL_REGISTRY[name]
    result = handler(**arguments)
    return {"content": [{"type": "text", "text": result}]}


# Register our tools with mock implementations
MOCK_USERS = {
    "alice@acme.com": {"name": "Alice", "role": "engineer", "tier": "premium"},
    "bob@acme.com": {"name": "Bob", "role": "manager", "tier": "enterprise"},
}


def lookup_user_handler(email: str) -> str:
    user = MOCK_USERS.get(email)
    if user:
        return f"Found: {user['name']}, role: {user['role']}, tier: {user['tier']}"
    return f"No user found with email: {email}"


def get_weather_handler(location: str) -> str:
    return f"Weather in {location}: 62°F, partly cloudy"


register_tool(lookup_tool, lookup_user_handler)
register_tool(weather_tool, get_weather_handler)

# Simulate tool calls
print("=== Tool Call: lookup_user ===")
result = call_mcp_tool("lookup_user", {"email": "alice@acme.com"})
print(json.dumps(result, indent=2))

print()
print("=== Tool Call: get_weather ===")
result = call_mcp_tool("get_weather", {"location": "San Francisco"})
print(json.dumps(result, indent=2))

print()
print("=== Tool Call: unknown tool ===")
result = call_mcp_tool("fly_to_moon", {})
print(json.dumps(result, indent=2))

=== Tool Call: lookup_user ===
{
  "content": [
    {
      "type": "text",
      "text": "Found: Alice, role: engineer, tier: premium"
    }
  ]
}

=== Tool Call: get_weather ===
{
  "content": [
    {
      "type": "text",
      "text": "Weather in San Francisco: 62\u00b0F, partly cloudy"
    }
  ]
}

=== Tool Call: unknown tool ===
{
  "error": "Unknown tool: fly_to_moon"
}


## Part 2: A2A — Agent Cards and Task Lifecycle

A2A operates at a different level. Instead of tools, we have **agents** that advertise their capabilities through **Agent Cards** and coordinate via **Tasks**.

In [5]:
from pydantic import BaseModel, Field
from enum import Enum


class AgentSkill(BaseModel):
    """A capability that an A2A agent advertises."""
    id: str
    name: str
    description: str
    tags: list[str] = Field(default_factory=list)
    examples: list[str] = Field(default_factory=list)


class AgentCapabilities(BaseModel):
    """Protocol-level capabilities the agent supports."""
    streaming: bool = False
    pushNotifications: bool = False


class AgentCard(BaseModel):
    """The A2A Agent Card — an agent's identity and capabilities."""
    name: str
    description: str
    url: str
    version: str = "1.0.0"
    capabilities: AgentCapabilities = Field(
        default_factory=AgentCapabilities
    )
    skills: list[AgentSkill] = Field(default_factory=list)
    provider: dict[str, str] = Field(default_factory=dict)


# Create agent cards for a multi-agent system
research_agent = AgentCard(
    name="Research Agent",
    description="Performs deep research on technical topics with citations",
    url="https://research-agent.internal",
    capabilities=AgentCapabilities(streaming=True, pushNotifications=True),
    skills=[
        AgentSkill(
            id="deep-research",
            name="Deep Research",
            description="Multi-source research with structured citations",
            tags=["research", "analysis"],
            examples=[
                "Research the latest advances in RLHF",
                "Compare transformer architectures for code generation",
            ],
        ),
        AgentSkill(
            id="summarize",
            name="Summarize",
            description="Produce a structured summary of a document",
            tags=["summarization"],
            examples=["Summarize this paper in 3 bullet points"],
        ),
    ],
    provider={"organization": "ML Team", "url": "https://ml.internal"},
)

billing_agent = AgentCard(
    name="Billing Agent",
    description="Handles billing inquiries, disputes, and account adjustments",
    url="https://billing-agent.internal",
    capabilities=AgentCapabilities(streaming=False, pushNotifications=True),
    skills=[
        AgentSkill(
            id="dispute-resolution",
            name="Dispute Resolution",
            description="Investigate and resolve billing disputes",
            tags=["billing", "dispute"],
            examples=["Customer was charged twice for subscription"],
        ),
    ],
    provider={"organization": "Finance Team"},
)

print("=== Research Agent Card ===")
print(json.dumps(research_agent.model_dump(), indent=2))

=== Research Agent Card ===
{
  "name": "Research Agent",
  "description": "Performs deep research on technical topics with citations",
  "url": "https://research-agent.internal",
  "version": "1.0.0",
  "capabilities": {
    "streaming": true,
    "pushNotifications": true
  },
  "skills": [
    {
      "id": "deep-research",
      "name": "Deep Research",
      "description": "Multi-source research with structured citations",
      "tags": [
        "research",
        "analysis"
      ],
      "examples": [
        "Research the latest advances in RLHF",
        "Compare transformer architectures for code generation"
      ]
    },
    {
      "id": "summarize",
      "name": "Summarize",
      "description": "Produce a structured summary of a document",
      "tags": [
        "summarization"
      ],
      "examples": [
        "Summarize this paper in 3 bullet points"
      ]
    }
  ],
  "provider": {
    "organization": "ML Team",
    "url": "https://ml.internal"
  }
}


### Agent Discovery

In A2A, agents discover each other by fetching Agent Cards. Let's simulate a registry.

In [6]:
class AgentRegistry:
    """Simple agent registry — in production this would be a service."""

    def __init__(self):
        self._agents: dict[str, AgentCard] = {}

    def register(self, card: AgentCard):
        self._agents[card.url] = card
        print(f"Registered: {card.name} at {card.url}")

    def discover_by_skill(self, tag: str) -> list[AgentCard]:
        """Find agents that have skills matching a tag."""
        matches = []
        for card in self._agents.values():
            for skill in card.skills:
                if tag in skill.tags:
                    matches.append(card)
                    break
        return matches

    def list_all(self) -> list[str]:
        return [f"{c.name} ({c.url})" for c in self._agents.values()]


# Build a registry
registry = AgentRegistry()
registry.register(research_agent)
registry.register(billing_agent)

print()
print("All agents:", registry.list_all())

print()
print("Agents with 'research' skill:")
for agent in registry.discover_by_skill("research"):
    print(f"  {agent.name}: {[s.name for s in agent.skills]}")

print()
print("Agents with 'billing' skill:")
for agent in registry.discover_by_skill("billing"):
    print(f"  {agent.name}: {[s.name for s in agent.skills]}")

Registered: Research Agent at https://research-agent.internal
Registered: Billing Agent at https://billing-agent.internal

All agents: ['Research Agent (https://research-agent.internal)', 'Billing Agent (https://billing-agent.internal)']

Agents with 'research' skill:
  Research Agent: ['Deep Research', 'Summarize']

Agents with 'billing' skill:
  Billing Agent: ['Dispute Resolution']


### A2A Task Lifecycle

A2A tasks progress through states: `PENDING → WORKING → COMPLETED/FAILED`. Let's model this and simulate the three execution patterns.

In [7]:
import uuid
import time


class TaskState(str, Enum):
    PENDING = "pending"
    WORKING = "working"
    INPUT_REQUIRED = "input_required"
    COMPLETED = "completed"
    FAILED = "failed"
    CANCELED = "canceled"


class TaskStatus(BaseModel):
    state: TaskState
    message: str = ""
    timestamp: float = Field(default_factory=time.time)


class Message(BaseModel):
    role: str  # "user" or "agent"
    parts: list[dict[str, str]]
    messageId: str = Field(default_factory=lambda: str(uuid.uuid4())[:8])


class Task(BaseModel):
    id: str = Field(default_factory=lambda: str(uuid.uuid4())[:8])
    status: TaskStatus = Field(
        default_factory=lambda: TaskStatus(state=TaskState.PENDING)
    )
    messages: list[Message] = Field(default_factory=list)
    artifacts: list[dict] = Field(default_factory=list)


# Create a task
task = Task()
task.messages.append(Message(
    role="user",
    parts=[{"kind": "text", "text": "Research RLHF alternatives"}],
))

print(f"Task {task.id} created")
print(f"  State: {task.status.state.value}")
print(f"  Messages: {len(task.messages)}")

Task 07a37b5e created
  State: pending
  Messages: 1


In [8]:
# Simulate the task progressing through states

def transition(task: Task, new_state: TaskState, message: str = ""):
    old_state = task.status.state
    task.status = TaskStatus(state=new_state, message=message)
    print(f"  {old_state.value} → {new_state.value}", end="")
    if message:
        print(f"  ({message})")
    else:
        print()


print(f"Task {task.id} lifecycle:")
transition(task, TaskState.WORKING, "Agent started processing")
transition(task, TaskState.WORKING, "Found 3 relevant papers")
transition(task, TaskState.WORKING, "Synthesizing findings")
transition(task, TaskState.COMPLETED, "Research complete")

# Add the result as an artifact
task.artifacts.append({
    "parts": [{
        "kind": "text",
        "text": "Key RLHF alternatives: DPO, KTO, ORPO..."
    }]
})

print(f"\nFinal state: {task.status.state.value}")
print(f"Artifacts: {len(task.artifacts)}")

Task 07a37b5e lifecycle:
  pending → working  (Agent started processing)
  working → working  (Found 3 relevant papers)
  working → working  (Synthesizing findings)
  working → completed  (Research complete)

Final state: completed
Artifacts: 1


### Three Execution Patterns

A2A supports synchronous (blocking), streaming, and async (polling) patterns. Let's simulate each.

In [9]:
import asyncio
from collections.abc import AsyncIterator


class MockA2AAgent:
    """Simulates an A2A agent that supports all three execution patterns."""

    def __init__(self, card: AgentCard):
        self.card = card
        self._tasks: dict[str, Task] = {}

    async def send_message_sync(self, text: str) -> Task:
        """Pattern 1: Synchronous — blocks until complete."""
        task = Task()
        task.messages.append(
            Message(role="user", parts=[{"kind": "text", "text": text}])
        )
        task.status = TaskStatus(
            state=TaskState.WORKING, message="Processing..."
        )
        # Simulate work
        await asyncio.sleep(0.1)
        task.status = TaskStatus(
            state=TaskState.COMPLETED, message="Done"
        )
        task.artifacts.append({
            "parts": [{"kind": "text", "text": f"Response to: {text}"}]
        })
        return task

    async def send_message_streaming(
        self, text: str
    ) -> AsyncIterator[dict]:
        """Pattern 2: Streaming — yields events as they happen."""
        task = Task()
        task.messages.append(
            Message(role="user", parts=[{"kind": "text", "text": text}])
        )
        self._tasks[task.id] = task

        yield {"type": "status", "state": "working", "msg": "Starting..."}
        await asyncio.sleep(0.05)

        yield {"type": "status", "state": "working", "msg": "Analyzing..."}
        await asyncio.sleep(0.05)

        yield {
            "type": "artifact",
            "text": f"Partial result for: {text}",
        }
        yield {"type": "status", "state": "completed", "msg": "Done"}

    async def send_message_async(self, text: str) -> str:
        """Pattern 3: Async — returns task ID for polling."""
        task = Task()
        task.messages.append(
            Message(role="user", parts=[{"kind": "text", "text": text}])
        )
        self._tasks[task.id] = task
        # Simulate background processing
        asyncio.create_task(self._process_async(task))
        return task.id

    async def _process_async(self, task: Task):
        task.status = TaskStatus(
            state=TaskState.WORKING, message="Processing in background"
        )
        await asyncio.sleep(0.2)
        task.status = TaskStatus(
            state=TaskState.COMPLETED, message="Done"
        )
        task.artifacts.append({
            "parts": [{
                "kind": "text",
                "text": "Async result ready"
            }]
        })

    async def get_task(self, task_id: str) -> Task | None:
        return self._tasks.get(task_id)

In [10]:
agent = MockA2AAgent(research_agent)

# Pattern 1: Synchronous
print("=== Pattern 1: Synchronous ===")
task = await agent.send_message_sync("What is DPO?")
print(f"State: {task.status.state.value}")
print(f"Result: {task.artifacts[0]['parts'][0]['text']}")

# Pattern 2: Streaming
print("\n=== Pattern 2: Streaming ===")
async for event in agent.send_message_streaming("Compare DPO vs RLHF"):
    if event["type"] == "status":
        print(f"  Status: {event['state']} - {event['msg']}")
    elif event["type"] == "artifact":
        print(f"  Artifact: {event['text']}")

# Pattern 3: Async with polling
print("\n=== Pattern 3: Async + Polling ===")
task_id = await agent.send_message_async("Full research on RLHF alternatives")
print(f"Task submitted: {task_id}")

for i in range(5):
    await asyncio.sleep(0.1)
    t = await agent.get_task(task_id)
    if t:
        print(f"  Poll {i+1}: {t.status.state.value} - {t.status.message}")
        if t.status.state == TaskState.COMPLETED:
            print(f"  Result: {t.artifacts[0]['parts'][0]['text']}")
            break

=== Pattern 1: Synchronous ===
State: completed
Result: Response to: What is DPO?

=== Pattern 2: Streaming ===
  Status: working - Starting...
  Status: working - Analyzing...


  Artifact: Partial result for: Compare DPO vs RLHF
  Status: completed - Done

=== Pattern 3: Async + Polling ===
Task submitted: b62b49dd
  Poll 1: working - Processing in background


  Poll 2: completed - Done
  Result: Async result ready


## Part 3: MCP + A2A Together

The real power comes from combining both protocols. An agent uses MCP to access tools, and A2A to collaborate with other agents.

In [11]:
class SupportAgent:
    """A support agent that uses MCP for tools and A2A for delegation."""

    def __init__(self, mcp_tools: dict, a2a_registry: AgentRegistry):
        self.mcp_tools = mcp_tools
        self.registry = a2a_registry

    def use_tool(self, tool_name: str, args: dict) -> dict:
        """Use an MCP tool to gather context."""
        print(f"  [MCP] Calling tool: {tool_name}({args})")
        return call_mcp_tool(tool_name, args)

    def find_specialist(self, tag: str) -> AgentCard | None:
        """Use A2A discovery to find a specialist agent."""
        agents = self.registry.discover_by_skill(tag)
        if agents:
            print(f"  [A2A] Found specialist: {agents[0].name}")
            return agents[0]
        print(f"  [A2A] No specialist found for '{tag}'")
        return None

    def handle_request(self, user_email: str, message: str):
        """Route a support request using MCP tools and A2A delegation."""
        print(f"\nHandling request from {user_email}: '{message}'")
        print("-" * 60)

        # Step 1: Use MCP to look up the customer
        customer = self.use_tool("lookup_user", {"email": user_email})
        customer_text = customer["content"][0]["text"]
        print(f"  → {customer_text}")

        # Step 2: Determine if we need a specialist
        needs_billing = any(
            kw in message.lower()
            for kw in ["billing", "charge", "invoice", "refund"]
        )
        needs_research = any(
            kw in message.lower()
            for kw in ["research", "compare", "analyze"]
        )

        if needs_billing:
            specialist = self.find_specialist("billing")
            if specialist:
                print(f"  [A2A] Delegating to {specialist.name}")
                print(f"  [A2A] Would send: '{message}' with context")
                print(f"  → Routed to billing specialist")
                return

        if needs_research:
            specialist = self.find_specialist("research")
            if specialist:
                print(f"  [A2A] Delegating to {specialist.name}")
                print(f"  [A2A] Would send: '{message}' with context")
                print(f"  → Routed to research specialist")
                return

        # Step 3: Handle directly
        print("  [Direct] Handling without delegation")
        print(f"  → Responding directly to {user_email}")


# Wire it up
support = SupportAgent(
    mcp_tools=TOOL_REGISTRY,
    a2a_registry=registry,
)

# Test different request types
support.handle_request(
    "alice@acme.com",
    "I was charged twice for my subscription"
)

support.handle_request(
    "bob@acme.com",
    "Can you research alternatives to our current LLM provider?"
)

support.handle_request(
    "alice@acme.com",
    "How do I reset my password?"
)


Handling request from alice@acme.com: 'I was charged twice for my subscription'
------------------------------------------------------------
  [MCP] Calling tool: lookup_user({'email': 'alice@acme.com'})
  → Found: Alice, role: engineer, tier: premium
  [A2A] Found specialist: Billing Agent
  [A2A] Delegating to Billing Agent
  [A2A] Would send: 'I was charged twice for my subscription' with context
  → Routed to billing specialist

Handling request from bob@acme.com: 'Can you research alternatives to our current LLM provider?'
------------------------------------------------------------
  [MCP] Calling tool: lookup_user({'email': 'bob@acme.com'})
  → Found: Bob, role: manager, tier: enterprise
  [A2A] Found specialist: Research Agent
  [A2A] Delegating to Research Agent
  [A2A] Would send: 'Can you research alternatives to our current LLM provider?' with context
  → Routed to research specialist

Handling request from alice@acme.com: 'How do I reset my password?'
--------------------

## Part 4: The Spectrum — From Tools to Skills to Multi-Agent

Let's visualize the architectural spectrum discussed in the post.

In [12]:
# The architectural spectrum from simple to complex
spectrum = [
    {
        "level": "Raw LLM",
        "context_unified": True,
        "tools": False,
        "specialization": "None",
        "coordination_overhead": "None",
        "feedback_loops": "Full (single context)",
        "use_when": "Simple Q&A, no external data needed",
    },
    {
        "level": "LLM + Tools (MCP)",
        "context_unified": True,
        "tools": True,
        "specialization": "Via tools",
        "coordination_overhead": "Minimal",
        "feedback_loops": "Full (single context)",
        "use_when": "Need external data/actions",
    },
    {
        "level": "LLM + Skills",
        "context_unified": True,
        "tools": True,
        "specialization": "Via skill routing",
        "coordination_overhead": "Low",
        "feedback_loops": "Full (single context)",
        "use_when": "Multiple domains, one agent",
    },
    {
        "level": "Multi-Agent (A2A)",
        "context_unified": False,
        "tools": True,
        "specialization": "Via separate agents",
        "coordination_overhead": "High",
        "feedback_loops": "Fragmented",
        "use_when": "Genuinely separate concerns/teams",
    },
]

# Print as a formatted table
headers = ["Level", "Unified Ctx", "Tools", "Specialization",
           "Overhead", "Feedback"]
rows = []
for s in spectrum:
    rows.append([
        s["level"],
        "Yes" if s["context_unified"] else "No",
        "Yes" if s["tools"] else "No",
        s["specialization"],
        s["coordination_overhead"],
        s["feedback_loops"],
    ])

# Calculate column widths
widths = [max(len(h), max(len(r[i]) for r in rows))
          for i, h in enumerate(headers)]

# Print header
header_line = " | ".join(h.ljust(w) for h, w in zip(headers, widths))
print(header_line)
print("-" * len(header_line))

# Print rows
for row in rows:
    print(" | ".join(v.ljust(w) for v, w in zip(row, widths)))

Level             | Unified Ctx | Tools | Specialization      | Overhead | Feedback             
------------------------------------------------------------------------------------------------
Raw LLM           | Yes         | No    | None                | None     | Full (single context)
LLM + Tools (MCP) | Yes         | Yes   | Via tools           | Minimal  | Full (single context)
LLM + Skills      | Yes         | Yes   | Via skill routing   | Low      | Full (single context)
Multi-Agent (A2A) | No          | Yes   | Via separate agents | High     | Fragmented           


## Takeaways

1. **MCP** is the context layer — use it to give your AI app access to tools and data (star topology)
2. **A2A** is the coordination layer — use it when agents need to discover and delegate to each other (mesh topology)
3. They're **complementary**, not competing — an A2A agent can use MCP internally
4. **Start simple**: Raw LLM → add MCP tools → add skills → go multi-agent only when needed
5. The **skills pattern** (single agent, multiple specialized capabilities) is the sweet spot for most teams

For more details, read the full blog post: [The Protocol Zoo](https://cmenguy.github.io/llm/ai-engineering/2026/01/28/agentic-protocols.html)